# 04 — Fair KnottedGraph vs Topoly Yamada comparison

This notebook runs identical-PD benchmarks. It does **not** use Topoly's misleading coordinate+bridges one-call path.

Correctness is checked before timing. The connected benchmark accepts only the standard Laurent convention transformations
$$
P_{\rm Topoly}(A)=\pm A^k P_{\rm KG}(A^{\pm1}),
$$
and records the sign, monomial shift and whether $A\leftrightarrow A^{-1}$ was required. Anything beyond these transformations is rejected.

Two suites are run: decomposable diagrams and connected spatial graphs.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, csv
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/"pyproject.toml").exists():
    ROOT=ROOT.parent
if not (ROOT/"src"/"knotted_graph").exists():
    raise RuntimeError("Run from inside the KnottedGraph checkout.")
SRC=ROOT/"src"
sys.path.insert(0,str(SRC))
branch=subprocess.check_output(["git","rev-parse","--abbrev-ref","HEAD"],cwd=ROOT,text=True).strip()
commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip()
print("ROOT   =",ROOT)
print("branch =",branch)
print("commit =",commit)
if branch!="perf/yamada-max-optimization":
    raise RuntimeError(f"Expected perf/yamada-max-optimization, got {branch}")
import knotted_graph
kg_path=Path(knotted_graph.__file__).resolve()
print("knotted_graph loaded from =",kg_path)
assert SRC in kg_path.parents, "A stale installed knotted_graph was imported."
try:
    import topoly
    print("topoly loaded from =",Path(topoly.__file__).resolve())
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT=ROOT/"User_guide"/"benchmarks"
RES=OUT/"results_latest"; FIG=OUT/"figures_latest"
RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)

In [ ]:
def run_summary(script,timeout=3600):
    path=ROOT/"dev"/script
    if not path.exists():
        raise FileNotFoundError(path)
    env=dict(os.environ)
    env["PYTHONPATH"]=str(SRC)
    env["PYTHONNOUSERSITE"]="1"
    p=subprocess.run([sys.executable,str(path)],cwd=ROOT,env=env,
                     text=True,capture_output=True,timeout=timeout)
    if p.stdout:
        print(p.stdout)
    if p.returncode:
        raise RuntimeError(
            f"{script} failed with exit code {p.returncode}.\n"
            f"STDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}"
        )
    for line in p.stdout.splitlines():
        if line.startswith("SUMMARY="):
            return json.loads(line[8:])
    raise RuntimeError(f"{script} completed but did not emit SUMMARY=")

decomp=run_summary("benchmark_topoly_identical_pd.py")
connected=run_summary("benchmark_topoly_connected_pd.py")
print(f"decomposable rows = {len(decomp)}")
print(f"connected rows    = {len(connected)}")

In [ ]:
def save(name,rows):
    keys=list(dict.fromkeys(k for r in rows for k in r))
    with (RES/name).open("w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=keys); w.writeheader(); w.writerows(rows)
    print("saved",RES/name)

save("04_topoly_identical_pd.csv",decomp)
save("04_topoly_connected_pd.csv",connected)

plt.figure(figsize=(8.5,5.3))
q=sorted(decomp,key=lambda r:r["crossings"])
plt.plot([r["crossings"] for r in q],[r["knottedgraph_s"] for r in q],marker="o",label="KnottedGraph")
plt.plot([r["crossings"] for r in q],[r["topoly_s"] for r in q],marker="o",label="Topoly")
plt.yscale("log"); plt.xlabel("Crossings c"); plt.ylabel("Yamada runtime (s)")
plt.title("Identical-PD Yamada benchmark"); plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
plt.savefig(FIG/"04_topoly_identical_pd.pdf",bbox_inches="tight")
plt.savefig(FIG/"04_topoly_identical_pd.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
print("Connected-graph results:")
for r in connected:
    print(r)
print("\nAll timings were accepted only after the benchmark scripts established Laurent-convention equivalence.")